In [ ]:
# ============================================================
# 02_SKILL_EXTRACTION.IPYNB
# Resume - Job Description Skill Extraction Module
# ============================================================

# ============================================================
# STEP 0 : IMPORT LIBRARIES
# ============================================================

import pandas as pd
import re
from google.colab import files

# ============================================================
# STEP 1 : LOAD CLEANED DATASET
# ============================================================

df = pd.read_csv("/cleaned_resumeJD2_pairs.csv")

print("="*50)
print("DATASET INFORMATION")
print("="*50)

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

# ============================================================
# STEP 2 : SKILL DATABASE
# ============================================================
#
# This skill database is used for dictionary-based skill extraction.
# It can be expanded later according to project requirements.
#
# ============================================================

SKILLS_DB = [

# ============================================================
# PROGRAMMING LANGUAGES
# ============================================================

"python","java","c++","c#","javascript","typescript",
"php","ruby","rust","kotlin","swift","matlab",

# ============================================================
# WEB DEVELOPMENT
# ============================================================

"html","css","react","angular","vue",
"nodejs","node.js",
"express","django","flask","fastapi",
"spring","spring boot",
"bootstrap","jquery",
"rest api","graphql",

# ============================================================
# DATABASES
# ============================================================

"sql","mysql","postgresql","mongodb",
"oracle","sqlite","redis",
"cassandra","dynamodb",

# ============================================================
# DATA SCIENCE / AI
# ============================================================

"machine learning",
"deep learning",
"artificial intelligence",
"nlp",
"natural language processing",
"computer vision",
"data science",
"data analysis",
"statistics",
"predictive modeling",

# Python Libraries

"numpy",
"pandas",
"matplotlib",
"seaborn",
"scikit learn",
"sklearn",
"tensorflow",
"keras",
"pytorch",
"xgboost",
"lightgbm",

# ============================================================
# CLOUD & DEVOPS
# ============================================================

"aws",
"azure",
"gcp",
"docker",
"kubernetes",
"jenkins",
"terraform",
"ansible",
"git",
"github",
"gitlab",
"ci cd",
"linux",
"unix",

# ============================================================
# DATA ENGINEERING
# ============================================================

"hadoop",
"spark",
"apache spark",
"hive",
"kafka",
"airflow",
"etl",

# ============================================================
# BUSINESS INTELLIGENCE
# ============================================================

"power bi",
"tableau",
"looker",
"excel",
"dashboarding",
"business intelligence",

# ============================================================
# FINANCE
# ============================================================

"cfa",
"ca",
"financial modeling",
"valuation",
"equity research",
"investment banking",
"financial reporting",
"accounting",
"treasury",
"budgeting",
"forecasting",
"quickbooks",
"sap",
"ifrs",
"gaap",
"financial analysis",
"fp&a",

# ============================================================
# HR
# ============================================================

"recruitment",
"talent acquisition",
"employee relations",
"payroll",
"performance management",
"human resources",
"onboarding",
"training and development",

# ============================================================
# MARKETING
# ============================================================

"digital marketing",
"seo",
"sem",
"google analytics",
"content marketing",
"email marketing",
"social media marketing",
"branding",
"market research",

# ============================================================
# PRODUCT MANAGEMENT
# ============================================================

"product management",
"roadmapping",
"product analytics",
"growth hacking",
"user research",
"a b testing",
"stakeholder management",

# ============================================================
# PROJECT MANAGEMENT
# ============================================================

"project management",
"agile",
"scrum",
"jira",
"kanban",
"risk management",

# ============================================================
# CYBER SECURITY
# ============================================================

"cyber security",
"penetration testing",
"ethical hacking",
"network security",
"information security",
"siem",

# ============================================================
# NETWORKING
# ============================================================

"tcp ip",
"dns",
"routing",
"switching",
"network administration",

# ============================================================
# SOFT SKILLS
# ============================================================

"communication",
"leadership",
"team management",
"problem solving",
"critical thinking",
"presentation skills",
"negotiation",
"customer service",

# ============================================================
# HEALTHCARE
# ============================================================

"medical billing",
"patient care",
"clinical research",
"healthcare management",

# ============================================================
# SALES
# ============================================================

"sales",
"business development",
"crm",
"salesforce",
"lead generation",
"customer relationship management"
]

print("\nTotal Skills in Database:", len(SKILLS_DB))

# ============================================================
# STEP 3 : SKILL NORMALIZATION
# ============================================================
#
# Different ways of writing the same skill
# will be standardized.
#
# Example:
# Chartered Accountant -> CA
#
# ============================================================

SKILL_ALIASES = {

    # Finance
    "chartered accountant": "ca",
    "certified public accountant": "ca",

    # HR
    "human resource": "human resources",

    # CRM
    "customer relationship management": "crm",

    # Marketing
    "search engine optimization": "seo",
    "search engine marketing": "sem",

    # FP&A
    "financial planning and analysis": "fp&a"
}

# ============================================================
# STEP 4 : NORMALIZATION FUNCTION
# ============================================================

def normalize_text(text):

    text = str(text).lower()

    for phrase, replacement in SKILL_ALIASES.items():

        text = text.replace(
            phrase,
            replacement
        )

    return text

# ============================================================
# STEP 5 : SKILL EXTRACTION FUNCTION
# ============================================================

def extract_skills(text):

    text = normalize_text(text)

    found_skills = []

    for skill in SKILLS_DB:

        pattern = r'\b' + re.escape(skill) + r'\b'

        if re.search(pattern, text):
            found_skills.append(skill)

    return sorted(
        list(set(found_skills))
    )

# ============================================================
# STEP 6 : EXTRACT RESUME SKILLS
# ============================================================

df["resume_skills"] = df["resume_text"].apply(
    extract_skills
)

print("\nResume Skill Extraction Completed")

# ============================================================
# STEP 7 : EXTRACT JD SKILLS
# ============================================================

df["jd_skills"] = df["job_description"].apply(
    extract_skills
)

print("JD Skill Extraction Completed")

# ============================================================
# STEP 8 : MATCHED SKILLS
# ============================================================

def get_matched_skills(
        resume_skills,
        jd_skills):

    return list(
        set(resume_skills)
        &
        set(jd_skills)
    )

# ============================================================
# STEP 9 : MISSING SKILLS
# ============================================================

def get_missing_skills(
        resume_skills,
        jd_skills):

    return list(
        set(jd_skills)
        -
        set(resume_skills)
    )

# ============================================================
# STEP 10 : SKILL MATCH SCORE
# ============================================================

def skill_match_score(
        resume_skills,
        jd_skills):

    if len(jd_skills) == 0:
        return 0

    matched = len(
        set(resume_skills)
        &
        set(jd_skills)
    )

    score = matched / len(jd_skills)

    return round(score, 2)

# ============================================================
# STEP 11 : CREATE NEW COLUMNS
# ============================================================

df["matched_skills"] = df.apply(
    lambda row:
    get_matched_skills(
        row["resume_skills"],
        row["jd_skills"]
    ),
    axis=1
)

df["missing_skills"] = df.apply(
    lambda row:
    get_missing_skills(
        row["resume_skills"],
        row["jd_skills"]
    ),
    axis=1
)

df["skill_match_score"] = df.apply(
    lambda row:
    skill_match_score(
        row["resume_skills"],
        row["jd_skills"]
    ),
    axis=1
)

# Additional useful columns

df["matched_skill_count"] = (
    df["matched_skills"]
    .apply(len)
)

df["missing_skill_count"] = (
    df["missing_skills"]
    .apply(len)
)

print("\nSkill Matching Completed")

# ============================================================
# STEP 12 : STATISTICS
# ============================================================

print("\n" + "="*50)
print("SKILL EXTRACTION STATISTICS")
print("="*50)

print(
    "\nRows with NO Resume Skills:",
    (df["resume_skills"].str.len() == 0).sum()
)

print(
    "Rows with NO JD Skills:",
    (df["jd_skills"].str.len() == 0).sum()
)

print("\nSkill Match Score Statistics:")

print(
    df["skill_match_score"]
    .describe()
)

# ============================================================
# STEP 13 : SAMPLE OUTPUT
# ============================================================

print("\n" + "="*50)
print("SAMPLE OUTPUT")
print("="*50)

print(

    df[
        [
            "resume_skills",
            "jd_skills",
            "matched_skills",
            "missing_skills",
            "skill_match_score"
        ]
    ].head(5)

)

# ============================================================
# STEP 14 : SAVE DATASET
# ============================================================

df.to_csv(
    "skill_extracted_resumeJD.csv",
    index=False
)

print("\nDataset Saved Successfully!")

# Download File

files.download(
    "skill_extracted_resumeJD.csv"
)

print(
    "\nFile Name: skill_extracted_resumeJD.csv"
)